# Aircraft campaigns: what was measured

An overview of the ARM G-1 aircraft data downloaded with `arrhenius_aircraft_download.sh`, before and alongside the Nd–CCN pairing in `aircraft_build.py`:

1. **Campaign table**: flights, flight hours, time in cloud, clear-air time, median droplet number and aerosol.
2. **Flight table** per campaign: one row per flight (date, times, altitude range, minutes in cloud, median Nd, near-base passes used).
3. **Flight tracks** on a map, coloured by altitude.
4. **Vertical profiles** of droplet number (in cloud) and aerosol (clear air).
5. **Size distributions**: median in-cloud droplet spectrum and median below-cloud aerosol spectrum, with IQR.
6. **Flight browser**: altitude, droplet number and aerosol through any single flight.
7. **Susceptibility results** from the built aircraft files, next to each other.

Definitions match `aircraft_build.py`: **in cloud** = droplet number (2–50 µm) ≥ 10 cm⁻³ and liquid water ≥ 0.05 g m⁻³; **clear air** = aerosol seconds that are not flagged and have no droplets (≥ 1 cm⁻³) within 2 s.

Run from the repository folder on Arrhenius (`data/aircraft/...` and `results/aircraft/...` must exist). Each campaign takes up to a minute to read; nothing is written to the data folders. Tables go to `results/aircraft/overview/`, figures to `figures/aircraft/overview/`.

In [ ]:
import sys, gc, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
warnings.filterwarnings("ignore", message="All-NaN slice")
warnings.filterwarnings("ignore", message="Mean of empty slice")

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
from aircraft_build import read_cloud, read_aerosol, _diam_um

DATA = ROOT / "data" / "aircraft"
RES = ROOT / "results" / "aircraft"
OUT = RES / "overview"
OUT.mkdir(parents=True, exist_ok=True)
FIG = ROOT / "figures" / "aircraft" / "overview" if (ROOT / "figures").is_dir() else OUT
FIG.mkdir(parents=True, exist_ok=True)

ND_RANGE = (2.0, 50.0)            # um, droplets (drizzle excluded)
IN_CLOUD = dict(nd=10.0, lwc=0.05)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

## Campaigns

`cloud` and `aerosol` are the datastream folders used (the merged products where they exist, else the single probes). `form` is passed to the aerosol reader (`auto` checks the file's own total; FIMS needs `dndlogdp`). `built` lists the output(s) of `aircraft_build.py` for section 7.

In [ ]:
CAMPAIGNS = {
    "ace-ena":  dict(label="ACE-ENA", region="Azores, marine Sc", when="Jun–Jul 2017, Jan–Feb 2018",
                     site=(39.0916, -28.0257), cloud="enaaafmergedcldsdF1.c1", aerosol="enaaafmergedaerosolsdF1.c1",
                     form="auto", built=["ace-ena_aircraft.nc"]),
    "hi-scale": dict(label="HI-SCALE", region="Oklahoma, continental Cu", when="Apr–May, Aug–Sep 2016",
                     site=(36.607, -97.488), cloud="sgpaafmergedcldsdF1.c1", aerosol="sgpaafmergedaerosolsdF1.c1",
                     form="auto", built=["hi-scale_aircraft.nc"]),
    "cacti":    dict(label="CACTI", region="Argentina, orographic", when="Nov–Dec 2018",
                     site=(-32.126, -64.728), cloud="coraafmergedcldsdF1.c1", aerosol="coraafmergedaerosolsdF1.c1",
                     form="auto", built=["cacti_aircraft.nc"]),
    "acapex":   dict(label="ACAPEX", region="California coast, marine", when="Jan–Mar 2015",
                     site=None, cloud="acxaaffcdpF1.c1", aerosol="acxaafuhsasF1.b1",
                     form="auto", built=["acapex_aircraft.nc"]),
    "acme-v":   dict(label="ACME-V", region="North Slope of Alaska, Arctic", when="Jun–Sep 2015",
                     site=(71.323, -156.609), cloud="nsaaaffcdpF1.c1", aerosol="nsaaafuhsasF1.b1",
                     form="auto", built=["acme-v_aircraft.nc"]),
    "goamazon": dict(label="GoAmazon", region="Amazon, Manaus plume", when="Feb–Mar, Sep–Oct 2014",
                     site=(-3.213, -60.598), cloud="maoaaffcdpF1.c1", aerosol="maoaafuhsasF1.b1",
                     form="auto", built=["goamazon_aircraft_uhsas.nc", "goamazon_aircraft.nc"]),
}
present = {k: v for k, v in CAMPAIGNS.items()
           if (DATA / k / v["cloud"]).is_dir() and any((DATA / k / v["cloud"]).iterdir())}
print("Campaigns with droplet data:", ", ".join(present) or "none")

## Read the data

For each campaign this keeps a compact 1-Hz table (time, position, altitude, droplet number, liquid water, in-cloud flag, aerosol total and N(>100 nm)), the median droplet and aerosol spectra, and drops the full size-distribution matrices to save memory.

In [ ]:
def droplet_spectrum(folder):
    """Median and IQR of in-cloud dN/dlogD (cm-3) over all in-cloud seconds of a campaign."""
    rows, dc = [], None
    for f in sorted(Path(folder).glob("*.nc")):
        with xr.open_dataset(f) as ds:
            b, dc = _diam_um(ds, "optical_diameter")
            n = ds["number_concentration"].values.astype(float)           # count/L/um
            n[n < 0] = np.nan
            w = np.clip(np.minimum(b[:, 1], ND_RANGE[1]) - np.maximum(b[:, 0], ND_RANGE[0]), 0, None)
            nd = np.nansum(n * w, axis=1) / 1000.0
            lwc = np.nansum(n * w * (np.pi / 6) * (dc * 1e-6) ** 3, axis=1) * 1e9
            inc = (nd >= IN_CLOUD["nd"]) & (lwc >= IN_CLOUD["lwc"])
            rows.append(n[inc] / 1000.0 * dc * np.log(10))               # -> dN/dlogD, cm-3
    a = np.vstack(rows) if rows else np.empty((0, 0))
    if not len(a):
        return None
    return pd.DataFrame({"D_um": dc, "median": np.nanmedian(a, 0), "q25": np.nanpercentile(a, 25, 0),
                         "q75": np.nanpercentile(a, 75, 0)})

def load_campaign(key, cfg):
    c = read_cloud(str(DATA / key / cfg["cloud"]), *ND_RANGE)
    c["in_cloud"] = (c["Nd"] >= IN_CLOUD["nd"]) & (c["LWC"] >= IN_CLOUD["lwc"])
    out = dict(cloud=c, spec_drop=droplet_spectrum(DATA / key / cfg["cloud"]), aer=None, spec_aer=None)
    afold = DATA / key / cfg["aerosol"]
    if afold.is_dir() and any(afold.iterdir()):
        a, am, dnm = read_aerosol(str(afold), cloud=c, form=cfg["form"])
        dlog = np.gradient(np.log10(dnm))
        arr = a.to_numpy()
        n100 = np.where(np.isfinite(arr[:, dnm >= 100]).mean(1) > 0.5, np.nansum(arr[:, dnm >= 100] * dlog[dnm >= 100], 1), np.nan)
        am = am.assign(N_tot=np.nansum(arr * dlog, 1), N100=n100)
        # below-cloud spectrum: clear-air seconds below the campaign's median in-cloud altitude
        zc = c.loc[c["in_cloud"], "alt"].median()
        sel = am["alt"].to_numpy() < zc if np.isfinite(zc) else np.ones(len(am), bool)
        if sel.sum() > 30:
            out["spec_aer"] = pd.DataFrame({"D_nm": dnm, "median": np.nanmedian(arr[sel], 0),
                                            "q25": np.nanpercentile(arr[sel], 25, 0),
                                            "q75": np.nanpercentile(arr[sel], 75, 0)})
        out["aer"] = am
        del a, arr
    gc.collect()
    return out

CAMP = {}
for k, cfg in present.items():
    print(f"\n===== {cfg['label']}")
    CAMP[k] = load_campaign(k, cfg)

## 1 · Campaign table

In [ ]:
def q(x, f="{:.0f}"):
    x = pd.Series(x).dropna()
    return f"{f.format(x.median())} [{f.format(x.quantile(.25))}–{f.format(x.quantile(.75))}]" if len(x) else "–"

def built_info(cfg):
    for name in cfg["built"]:
        p = RES / name
        if p.exists():
            with xr.open_dataset(p) as ds:
                return name, ds.sizes["time"], pd.DatetimeIndex(ds["time"].values).normalize().nunique()
    return None, 0, 0

rows = []
for k, d in CAMP.items():
    cfg, c, a = CAMPAIGNS[k], d["cloud"], d["aer"]
    flights = c.index.normalize().unique()
    built, npass, nfl = built_info(cfg)
    rows.append({"campaign": cfg["label"], "region": cfg["region"], "period": cfg["when"],
                 "flights": len(flights), "cloud-probe hours": round(len(c) / 3600, 1),
                 "in-cloud minutes": int(c["in_cloud"].sum() / 60),
                 "clear-air aerosol hours": round(len(a) / 3600, 1) if a is not None else 0,
                 "Nd in cloud, cm-3": q(c.loc[c["in_cloud"], "Nd"]),
                 "aerosol N_tot, cm-3": q(a["N_tot"]) if a is not None else "–",
                 "N(>100 nm), cm-3": q(a["N100"]) if a is not None else "–",
                 "near-base passes paired": npass, "flights with passes": nfl,
                 "cloud probe": cfg["cloud"], "aerosol probe": cfg["aerosol"]})
camp_table = pd.DataFrame(rows).set_index("campaign")
camp_table.to_csv(OUT / "campaigns.csv")
camp_table

## 2 · Flight tables

One row per flight (UTC date). Saved per campaign as `flights_<campaign>.csv`.

In [ ]:
FLIGHTS = {}
for k, d in CAMP.items():
    c, a = d["cloud"], d["aer"]
    built = next((RES / n for n in CAMPAIGNS[k]["built"] if (RES / n).exists()), None)
    passes = pd.Series(dtype=int)
    if built is not None:
        with xr.open_dataset(built) as ds:
            passes = pd.Series(1, index=pd.DatetimeIndex(ds["time"].values).normalize()).groupby(level=0).size()
    rows = []
    for day, g in c.groupby(c.index.normalize()):
        ga = a[a.index.normalize() == day] if a is not None else None
        rows.append({"date": day.date(), "start UTC": g.index.min().strftime("%H:%M"), "end UTC": g.index.max().strftime("%H:%M"),
                     "hours": round((g.index.max() - g.index.min()).total_seconds() / 3600, 1),
                     "alt min–max, m": f"{g['alt'].min():.0f}–{g['alt'].max():.0f}",
                     "in-cloud min": round(g["in_cloud"].sum() / 60, 1),
                     "median Nd in cloud": round(g.loc[g["in_cloud"], "Nd"].median(), 0) if g["in_cloud"].any() else np.nan,
                     "clear-air aerosol min": round(len(ga) / 60, 1) if ga is not None else 0,
                     "median N_tot": round(ga["N_tot"].median(), 0) if ga is not None and len(ga) else np.nan,
                     "passes paired": int(passes.get(day, 0))})
    FLIGHTS[k] = pd.DataFrame(rows).set_index("date")
    FLIGHTS[k].to_csv(OUT / f"flights_{k}.csv")
    print(f"\n{CAMPAIGNS[k]['label']}: {len(rows)} flights")
    display(FLIGHTS[k])

## 3 · Flight tracks

Every 10th second, coloured by altitude. The star marks the ARM ground site.

In [ ]:
keys = list(CAMP)
ncol = min(3, len(keys)); nrow = int(np.ceil(len(keys) / ncol))
fig, axs = plt.subplots(nrow, ncol, figsize=(5 * ncol, 4.2 * nrow), squeeze=False)
for ax, k in zip(axs.flat, keys):
    c = CAMP[k]["cloud"].iloc[::10]
    sc = ax.scatter(c["lon"], c["lat"], c=c["alt"], s=1, cmap="viridis", vmin=0, vmax=np.nanpercentile(c["alt"], 98))
    if CAMPAIGNS[k]["site"]:
        ax.plot(CAMPAIGNS[k]["site"][1], CAMPAIGNS[k]["site"][0], "*", color="red", ms=12, mec="k")
    ax.set_title(f"{CAMPAIGNS[k]['label']} ({c.index.normalize().nunique()} flights)", loc="left", fontsize=10)
    ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
    ax.set_aspect(1 / np.cos(np.deg2rad(np.nanmedian(c["lat"]))))
    fig.colorbar(sc, ax=ax, label="altitude, m", shrink=0.8)
for ax in list(axs.flat)[len(keys):]:
    ax.set_visible(False)
fig.tight_layout()
fig.savefig(FIG / "flight_tracks.png", dpi=150)
plt.show(); plt.close(fig)

## 4 · Vertical profiles

Median (line) and IQR (shading) in 100 m altitude bins (above sea level): droplet number in cloud, and aerosol total and N(>100 nm) in clear air. Bins with fewer than 30 s are left out.

In [ ]:
zb = np.arange(0, 4100, 100)
def profile(alt, x, min_n=30):
    s = pd.DataFrame({"z": pd.cut(alt, zb, labels=zb[:-1] + 50), "x": x}).dropna()
    g = s.groupby("z", observed=True)["x"]
    p = pd.DataFrame({"med": g.median(), "q25": g.quantile(.25), "q75": g.quantile(.75), "n": g.size()})
    return p[p["n"] >= min_n]

fig, axs = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
for k, col in zip(keys, plt.cm.tab10.colors):
    c, a = CAMP[k]["cloud"], CAMP[k]["aer"]
    lab = CAMPAIGNS[k]["label"]
    sets = [(axs[0], profile(c.loc[c.in_cloud, "alt"], c.loc[c.in_cloud, "Nd"]))]
    if a is not None:
        sets += [(axs[1], profile(a["alt"], a["N_tot"])), (axs[2], profile(a["alt"], a["N100"]))]
    for ax, p in sets:
        if len(p):
            z = p.index.astype(float)
            ax.fill_betweenx(z, p["q25"], p["q75"], color=col, alpha=0.2, lw=0)
            ax.plot(p["med"], z, color=col, lw=1.8, label=lab)
for ax, t in zip(axs, ["Droplet number in cloud, cm$^{-3}$", "Aerosol N$_{tot}$, clear air, cm$^{-3}$", "N(D > 100 nm), clear air, cm$^{-3}$"]):
    ax.set_xscale("log"); ax.set_xlabel(t)
axs[0].set_ylabel("altitude, m above sea level"); axs[0].legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "vertical_profiles.png", dpi=150)
plt.show(); plt.close(fig)

## 5 · Size distributions

Left: median in-cloud droplet spectrum (all in-cloud seconds, drizzle range shown but excluded from Nd). Right: median clear-air aerosol spectrum below each campaign's median in-cloud altitude. Shading is the IQR. Note the aerosol probes differ (merged FIMS+PCASP from 10 nm; UHSAS from ~60 nm).

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4.8))
for k, col in zip(keys, plt.cm.tab10.colors):
    sd, sa = CAMP[k]["spec_drop"], CAMP[k]["spec_aer"]
    lab = CAMPAIGNS[k]["label"]
    if sd is not None:
        ok = sd["median"] > 0
        axs[0].fill_between(sd["D_um"], sd["q25"].clip(lower=1e-6), sd["q75"], color=col, alpha=0.15, lw=0)
        axs[0].plot(sd["D_um"][ok], sd["median"][ok], color=col, lw=1.8, label=lab)
    if sa is not None:
        axs[1].fill_between(sa["D_nm"], sa["q25"].clip(lower=1e-3), sa["q75"], color=col, alpha=0.15, lw=0)
        axs[1].plot(sa["D_nm"], sa["median"], color=col, lw=1.8, label=lab)
axs[0].axvspan(ND_RANGE[1], 2e4, color="0.9", zorder=0)
axs[0].set(xscale="log", yscale="log", xlim=(1, 2000), ylim=(1e-4, None),
           xlabel="Droplet diameter, µm", ylabel="dN/dlogD, cm$^{-3}$", title="In-cloud droplets")
axs[1].set(xscale="log", yscale="log", xlim=(10, 3000), ylim=(1, None),
           xlabel="Particle diameter, nm", ylabel="dN/dlogD$_p$, cm$^{-3}$", title="Below-cloud aerosol, clear air")
for ax in axs:
    ax.title.set_ha("left"); ax.title.set_x(0)
axs[0].legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "size_distributions.png", dpi=150)
plt.show(); plt.close(fig)

## 6 · Flight browser

`plot_flight("ace-ena", "2017-07-15")` shows one flight: altitude (in-cloud seconds highlighted, near-base passes from the built file marked), droplet number, and clear-air aerosol. Dates are in the flight tables above. The cell below shows the flight with the most paired passes in each campaign.

In [ ]:
def plot_flight(key, date, save=True):
    day = pd.Timestamp(date).normalize()
    c = CAMP[key]["cloud"]; c = c[c.index.normalize() == day]
    a = CAMP[key]["aer"]; a = a[a.index.normalize() == day] if a is not None else None
    if c.empty:
        print(f"No cloud-probe data for {key} on {day.date()}"); return
    fig, axs = plt.subplots(3, 1, figsize=(12, 7.5), sharex=True, gridspec_kw=dict(height_ratios=[1.3, 1, 1]))
    axs[0].plot(c.index, c["alt"], color="0.6", lw=0.8, label="altitude")
    axs[0].plot(c.index[c.in_cloud], c.loc[c.in_cloud, "alt"], ".", ms=2, color="#3B6FB6", label="in cloud")
    built = next((RES / n for n in CAMPAIGNS[key]["built"] if (RES / n).exists()), None)
    if built is not None:
        with xr.open_dataset(built) as ds:
            t = pd.DatetimeIndex(ds["time"].values); s = t.normalize() == day
            if s.any():
                axs[0].plot(t[s], ds["alt"].values[s], "v", color="#D9822B", ms=6, label="paired pass")
                axs[0].plot(t[s], ds["cloud_base"].values[s], "_", color="k", ms=12, mew=2, label="local cloud base")
    axs[0].set_ylabel("altitude, m"); axs[0].legend(frameon=False, fontsize=8, ncol=4, loc="upper right")
    axs[1].semilogy(c.index[c.in_cloud], c.loc[c.in_cloud, "Nd"], ".", ms=2, color="#3B6FB6")
    axs[1].set_ylabel("Nd in cloud, cm$^{-3}$")
    if a is not None and len(a):
        axs[2].semilogy(a.index, a["N_tot"], ".", ms=1.5, color="0.4", label="N$_{tot}$")
        axs[2].semilogy(a.index, a["N100"], ".", ms=1.5, color="#D9822B", label="N(>100 nm)")
        axs[2].legend(frameon=False, fontsize=8, markerscale=5)
    axs[2].set_ylabel("clear-air aerosol, cm$^{-3}$"); axs[2].set_xlabel("UTC")
    axs[0].set_title(f"{CAMPAIGNS[key]['label']} flight {day.date()}", loc="left")
    fig.tight_layout()
    if save:
        fig.savefig(FIG / f"flight_{key}_{day:%Y%m%d}.png", dpi=150)
    plt.show(); plt.close(fig)

for k in keys:
    ft = FLIGHTS[k]
    best = ft["passes paired"].idxmax() if ft["passes paired"].max() > 0 else ft["in-cloud min"].idxmax()
    plot_flight(k, best)

## 7 · Susceptibility from the aircraft

The quicklook summaries of the built aircraft files (run `arm_ena_nd_pnsd.py quicklook <file> --min-cloud-frac 0.5` first; each "hour" is a near-base cloud pass and each "day" a flight).

In [ ]:
import re
rows = []
for k, cfg in CAMPAIGNS.items():
    for name in cfg["built"]:
        nc = RES / name
        txt = nc.with_name(nc.stem + "_quicklook.txt")
        if not nc.exists():
            continue
        out = txt.read_text(errors="replace") if txt.exists() else ""
        g = lambda p: (re.search(p, out).groups() if re.search(p, out) else None)
        n = g(r"Using cloud_frac >= [\d.]+: (\d+) hours on (\d+) days")
        S = g(r"S\(100 nm\) = (-?[\d.]+|nan)\s+95% CI \[(-?[\d.]+|nan), (-?[\d.]+|nan)\]")
        D = g(r"D_Delta = (\d+) nm\s+95% CI \[(\d+), (\d+)\]")
        rows.append({"campaign": cfg["label"], "file": name, "passes": int(n[0]) if n else None,
                     "flights": int(n[1]) if n else None,
                     "S(100 nm)": f"{S[0]} [{S[1]}, {S[2]}]" if S else ("run quicklook" if not out else "–"),
                     "closure diameter, nm": f"{D[0]} [{D[1]}, {D[2]}]" if D else ("no crossing" if S else "–")})
res = pd.DataFrame(rows).set_index("campaign") if rows else pd.DataFrame()
res.to_csv(OUT / "aircraft_susceptibility.csv")
res

## Notes

- **Altitudes** are above sea level. ACE-ENA, ACAPEX and ACME-V fly over sea or flat coast; HI-SCALE over ground at ~315 m; CACTI over terrain of 1–2 km, so "low" there is relative to rising ground.
- **Flights** are grouped by UTC date; a flight crossing midnight UTC appears as two.
- **Single-probe campaigns** (ACAPEX, ACME-V, GoAmazon) have no cloud flag on the aerosol probe; clear air is defined from the droplet probe (no droplets ≥ 1 cm⁻³ within 2 s).
- **ACME-V UHSAS** only covers Aug–Oct 2015; switch `aerosol` to `nsaaafpcaspF1.b1` for the full campaign (from ~110 nm).
- **The droplet spectra** include drizzle sizes for completeness; droplet number everywhere in this project uses 2–50 µm only.